# Lekcja 3.3 — eksperymenty: funkcje oceny i ograniczona głębokość

Uruchamiaj komórki po kolei. Przed każdym eksperymentem zapisz swoje przewidywanie w komórce „Twoje przewidywanie” (kliknij ją dwukrotnie, żeby edytować), a dopiero potem uruchom kod.

Pierwsza komórka wybiera implementację:

- `"mine"` używa Twojego `task.py`; najpierw rozwiąż część B zadania;
- `"reference"` używa implementacji wzorcowej kursu. Wpisz `IMPLEMENTATION = "reference"`, jeśli chcesz uruchomić notebook przed skończeniem zadania.

W trybie `"reference"` w miejscu Twojej funkcji oceny występuje ocena przez linie ważone.

In [ ]:
%load_ext autoreload
%autoreload 2
import os

# "mine" uses your task.py; "reference" uses the course's reference implementation.
IMPLEMENTATION = os.environ.get("INTROAI_IMPLEMENTATION", "mine")
if IMPLEMENTATION == "mine":
    import task as algorithms
else:
    from introai.games import reference as algorithms

Funkcja `run` uruchamia jedno przeszukiwanie i zwraca jego wynik, liczbę odwiedzonych pozycji i czas w sekundach. Funkcja `player(depth, evaluate)` tworzy gracza, który wybiera ruchy Twoim przeszukiwaniem z podaną głębokością i oceną. Przeciwnik w turnieju to wzorcowe przeszukiwanie na głębokość 2 z oceną, która zawsze zwraca 0, tak jak w teście meczu. Oceny przez linie otwarte i przez linie ważone pochodzą z wersji wzorcowej kursu.

In [ ]:
import time

import matplotlib.pyplot as plt

from introai.games import KInARow, reference
from introai.games.players import match, search_player
from introai.testing.games import positions_visited
from introai.trace import TraceLimitExceeded, record

HINT = 'check task.py (or set IMPLEMENTATION = "reference" in the first cell)'
BOARD = KInARow(5, 5, 4)
if IMPLEMENTATION == "mine":
    your_evaluate = algorithms.evaluate
else:
    your_evaluate = reference.weighted_lines


def run(search, game, state, *args, limit=1_000_000):
    """Run one search; return its result, the positions visited and the time in seconds."""
    start = time.perf_counter()
    try:
        with record(limit=limit, kinds={"result"}) as trace:
            result = search(game, state, *args)
    except NotImplementedError as error:
        raise RuntimeError(str(error)) from None
    except (TraceLimitExceeded, RecursionError):
        message = f"{search.__name__} visited more than {limit} positions - {HINT}"
        raise RuntimeError(message) from None
    return result, positions_visited(trace), time.perf_counter() - start


def player(depth, evaluate):
    """A player that chooses moves with your depth-limited search."""

    def choose(game, state):
        (_, move), _, _ = run(algorithms.h_alpha_beta_search, game, state, depth, evaluate)
        return move

    return choose


opponent = search_player(reference.h_alpha_beta_search, depth=2, evaluate=reference.zero_evaluation)

## Eksperyment 1: turniej na planszy 5 × 5

Cztery funkcje oceny, każda z głębokością 1, 2 i 3, rozgrywają ten sam mecz co w teście: 12 partii z 6 otwarć, każde krzyżykami i kółkami, przeciw przeszukiwaniu na głębokość 2 bez oceny. W tabeli są wygrane, remisy i porażki. Obliczenia trwają kilka sekund.

**Twoje przewidywanie.** Która ocena wypadnie najlepiej? Czy ocena zerowa z głębokością 3 pokona przeciwnika z głębokością 2? Czy większa głębokość zawsze pomaga?

*Moje przewidywanie:* …

In [ ]:
evaluations = [
    ("zero", reference.zero_evaluation),
    ("open lines", reference.open_lines),
    ("weighted lines", reference.weighted_lines),
    ("yours", your_evaluate),
]
start = time.perf_counter()
print(f"{'evaluation':<16}{'depth 1':>10}{'depth 2':>10}{'depth 3':>10}   (wins/draws/losses)")
for name, evaluate in evaluations:
    row = []
    for depth in (1, 2, 3):
        table = match(BOARD, player(depth, evaluate), opponent)
        row.append(f"{table['win']}/{table['draw']}/{table['loss']}")
    print(f"{name:<16}{row[0]:>10}{row[1]:>10}{row[2]:>10}")
print(f"\n{time.perf_counter() - start:.1f} s")

**Do przemyślenia.** Jak wypadła ocena zerowa z głębokością 1, a jak z głębokością 3? Co to mówi o tym, czy lepiej mieć dobrą ocenę, czy głębsze przeszukiwanie? Czy Twoja ocena wygrywa także przy głębokości 1 i 3, czy tylko przy 2, którą sprawdza test?

## Eksperyment 2: pozycja z groźbą

X ma ruch. O ma dwa kółka w pierwszej kolumnie, a pola nad nimi i pod nimi są wolne. Przeszukiwanie wybiera ruch na głębokości od 1 do 4, raz z oceną przez linie ważone, raz z Twoją.

**Twoje przewidywanie.** Który ruch powinien zagrać X i dlaczego? Od jakiej głębokości przeszukiwanie go znajdzie?

*Moje przewidywanie:* …

In [ ]:
threat = BOARD.from_rows([".....", "O....", "O..X.", ".X...", "....."])
print(BOARD.render(threat))
print()
print(f"{'depth':<7}{'weighted lines':>24}{'yours':>24}")
for depth in (1, 2, 3, 4):
    row = []
    for evaluate in (reference.weighted_lines, your_evaluate):
        (value, move), _, _ = run(algorithms.h_alpha_beta_search, BOARD, threat, depth, evaluate)
        row.append(f"{move!s:>10} {value:+.3f}")
    print(f"{depth:<7}{row[0]:>24}{row[1]:>24}")

**Do przemyślenia.** Dlaczego przeszukiwanie na głębokość 1 nie widzi niebezpieczeństwa? Spójrz też na oceny: dlaczego przy głębokościach parzystych są wyraźnie niższe niż przy nieparzystych? Kto ma ostatni ruch przed horyzontem w każdym z tych przypadków?

## Eksperyment 3: koszt głębokości

Twoje przeszukiwanie z Twoją oceną szuka pierwszego ruchu na pustej planszy 5 × 5 z głębokością od 1 do 4. Mierzymy, ile pozycji odwiedza i ile trwa ten jeden ruch. Obie osie pionowe mają skalę logarytmiczną.

**Twoje przewidywanie.** Ile razy więcej pozycji kosztuje każdy kolejny półruch? Ile musiałby trwać ruch na głębokość 8?

*Moje przewidywanie:* …

In [ ]:
depths = [1, 2, 3, 4]
positions, seconds = [], []
for depth in depths:
    _, visited, elapsed = run(
        algorithms.h_alpha_beta_search, BOARD, BOARD.initial, depth, your_evaluate
    )
    positions.append(visited)
    seconds.append(elapsed)
    print(f"depth {depth}: {visited:>7} positions, {elapsed:.3f} s")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, values, title in (
    (axes[0], positions, "Positions visited for one move (log scale)"),
    (axes[1], seconds, "Seconds for one move (log scale)"),
):
    ax.plot(
        depths,
        values,
        color="#2a78d6",
        linewidth=2,
        marker="o",
        markersize=8,
        markeredgecolor="#fcfcfb",
        markeredgewidth=1.5,
    )
    ax.set_yscale("log")
    ax.set_xticks(depths)
    ax.set_title(title, loc="left", color="#0b0b0b")
    ax.set_xlabel("search depth (plies)", color="#52514e")
    ax.grid(axis="y", color="#e1e0d9", linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color("#c3c2b7")
    ax.tick_params(colors="#52514e")
fig.tight_layout()

**Do przemyślenia.** Jaki jest stosunek liczby pozycji między kolejnymi głębokościami? Porównaj go z liczbą ruchów na planszy (25, potem 24, …) i z jej pierwiastkiem. Jak głęboko zdążyłby przeszukać Twój program w ciągu jednej sekundy?